**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Image Processing as 2-D DSP

Images are just signals with two time axes. Everything from [Foundations](./Foundations_of_Signal_Processing_1.ipynb) transfers — convolution, spectra, sampling — and the payoff is a straight bridge into [CNNs](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

## 1. Pre-requisites

[Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (DFT & sampling), [Filter Design](./Filter_Design.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# Synthesize a test image (no downloads, fully reproducible):
# a "room": gradient wall, striped rug, circular table, textured window
yy, xx = np.mgrid[0:512, 0:512] / 512
img = 0.35 + 0.3 * yy                                        # lit wall
img += 0.25 * ((np.sin(2*np.pi*18*xx) > 0) & (yy > 0.72))    # striped rug
img += 0.3 * (((xx-0.68)**2 + (yy-0.42)**2) < 0.03)          # table disc
win = (np.abs(xx-0.22) < 0.13) & (np.abs(yy-0.28) < 0.16)
img += win * (0.25 + 0.12 * np.sin(2*np.pi*40*(xx+yy)))      # textured window
img = np.clip(img + 0.02 * rng.standard_normal((512, 512)), 0, 1)

plt.figure(figsize=(3.4, 3.4)); plt.imshow(img, cmap="gray"); plt.axis("off")
plt.title("our (synthetic) test subject"); plt.tight_layout(); plt.show()

/tmp/ipykernel_2032497/789045588.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("our (synthetic) test subject"); plt.tight_layout(); plt.show()


---
### 🕐 Session 1 of 3 — *2-D Convolution & the 2-D Spectrum* (~35 min)
**Goal:** extend filtering and Fourier to two dimensions; learn to read a 2-D spectrum.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb). &nbsp; **Feeds into:** Session 2 (sampling & aliasing).

---

## 2. Two Axes, Same Rules

💡 **Intuition.** 2-D convolution slides a small *kernel* over the image — identical mechanics to 1-D, once per axis. The 2-D DFT decomposes an image into **plane waves**: gratings of every orientation and spatial frequency. Reading the spectrum: center = DC (average brightness), distance from center = fineness of detail, *direction* = orientation of the stripes producing it. Edges in direction θ light up spectrum energy perpendicular to θ.

In [2]:
# Blur = 2-D low-pass; the spectrum shows exactly what was removed
k = np.outer(np.hanning(15), np.hanning(15)); k /= k.sum()
blurred = sig.convolve2d(img, k, mode="same", boundary="symm")

def spec2d(im):
    return np.log10(np.abs(np.fft.fftshift(np.fft.fft2(im))) + 1e-3)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, (im, ttl) in zip(axes, [(img, "image"), (spec2d(img), "its spectrum"),
                                 (blurred, "blurred"), (spec2d(blurred), "spectrum: highs gone")]):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl); ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2032497/2807211250.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Sampling, Aliasing & Moiré* (~35 min)
**Goal:** see 2-D aliasing with your own eyes; fix it the honest way.
**Builds on:** Session 1; [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) S2. &nbsp; **Feeds into:** Session 3 (edges).

---

## 3. Aliasing You Can See

💡 **Intuition.** Downsampling an image without low-passing folds fine patterns into coarse fake ones — **moiré**. It's the wagon-wheel effect with a second axis: the shirt stripes on TV that swim with rainbow bands are spatial frequencies beyond the sensor's Nyquist, aliased into visibility. The cure is the same as [1-D decimation](./Foundations_of_Signal_Processing_2.ipynb): blur (anti-alias filter) *before* subsampling — cameras do it optically.

In [3]:
# A radial chirp: frequency grows outward — the classic aliasing stress test
yy, xx = np.mgrid[-1:1:512j, -1:1:512j]
r2 = xx**2 + yy**2
zone = np.cos(2 * np.pi * 48 * r2)                    # 'zone plate'

M = 4
naive = zone[::M, ::M]
pre = sig.convolve2d(zone, np.ones((M, M))/M**2, mode="same", boundary="symm")
proper = pre[::M, ::M]

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3))
for ax, (im, ttl) in zip(axes, [(zone, "zone plate (freq ↑ outward)"),
                                 (naive, "naive ↓4: fake rings = moiré"),
                                 (proper, "blur first: honest, just softer")]):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2032497/3362991745.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Edges → Learned Features* (~40 min)
**Goal:** hand-designed gradient kernels, then the straight line to CNNs.
**Builds on:** Session 2.

---

## 4. Edge Detection

💡 **Intuition.** An edge is a spatial derivative — and differentiation is a high-pass filter. The Sobel kernels estimate the gradient along each axis while smoothing along the other; magnitude gives edge strength, arctangent gives orientation. Every classical vision pipeline (and the *first layer of every trained CNN*, as you saw in the [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb)) starts with filters of exactly this shape — the difference is that CNNs *learn* theirs.

In [4]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
gx = sig.convolve2d(img, sobel_x, mode="same", boundary="symm")
gy = sig.convolve2d(img, sobel_x.T, mode="same", boundary="symm")
mag = np.hypot(gx, gy)
orient = np.arctan2(gy, gx)

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3))
axes[0].imshow(mag, cmap="gray"); axes[0].set_title("gradient magnitude: edges")
axes[1].imshow(np.where(mag > 0.6, orient, np.nan), cmap="hsv"); axes[1].set_title("orientation (hue)")
axes[2].imshow((mag > np.quantile(mag, 0.93)), cmap="gray"); axes[2].set_title("thresholded edge map")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2032497/3243686540.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# Unsharp masking: the 100-year-old trick still inside every photo app
detail = img - blurred                       # what the blur removed
sharpened = np.clip(img + 1.2 * detail, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
axes[0].imshow(img[100:300, 100:300], cmap="gray"); axes[0].set_title("original")
axes[1].imshow(sharpened[100:300, 100:300], cmap="gray"); axes[1].set_title("unsharp masked: original + 1.2×(high-pass)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2032497/2876710011.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Same theory, second axis: plane-wave spectra, moiré as visible aliasing, edges as gradients, sharpening as boosted high-pass. When the [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) learns its first layer, it rediscovers this session.

---
## Where next

- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — kernels chosen by gradient descent instead of by Sobel.
- [Compressed Sensing](./Compressed_Sensing.ipynb) — images from far fewer samples than pixels.